# Report — Nowcast da abertura USDBRL (proxy do fixing) pré-mercado

Consolida os resultados do pipeline `src/` (rodar antes: scripts 01→04).

**Especificação (consenso López de Prado × Simons, ver `docs/debate_transcript.md`):**
- Target: `y = log(abertura Bloomberg ~9:00) − log(dolar_cc às 8:50)` em bps
- 1 linha/dia útil, snapshot point-in-time 8:50 BRT, features disponíveis em tempo real
- Validação: walk-forward expandindo (60d, passo 5, embargo 2), benchmarks b1/b2, Diebold-Mariano
- Ablação USDT/BRL (ressalva do usuário: regime de spread de 2024 ≠ atual)

In [ ]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
df = pd.read_csv('../data/processed/dataset_daily.csv', parse_dates=['date']).set_index('date')
y = df['y_log_resid']*1e4
print(f"Dataset: {len(df)} dias | {df.index[0].date()} -> {df.index[-1].date()}")
print(f"Resíduo (bps): média={y.mean():+.1f}, sd={y.std():.1f}")
df[['open_bloom','dolar_cc_snap','y_log_resid','ret_fut_on','ret_dxy_on','vol_6l_on',
    'event_day','basis_lag1','us_gdp_real_last','br_pib_real_last']].tail()

## 1. O resíduo (o que o modelo precisa explicar)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
y.plot(ax=ax[0], marker='.', lw=.6, title='Resíduo diário: abertura vs dolar_cc 8:50 (bps)')
ax[0].axhline(0, color='k', lw=.5)
ax[0].axhline(y.mean(), color='r', ls='--', lw=.8, label=f'média {y.mean():+.1f}')
ax[0].legend()
y.hist(bins=40, ax=ax[1]); ax[1].set_title('Distribuição (bps)')
plt.tight_layout(); plt.show()

## 2. Diagnósticos (gerados pelo `02_diagnostics.py`)

In [ ]:
print(open('../results/diagnostics_report.txt', encoding='utf-8').read())

## 3. Walk-forward: modelo vs benchmarks

In [ ]:
reg = json.load(open('../results/configs_registry.json', encoding='utf-8'))
res = pd.DataFrame(reg['atuais']).T
print(res.to_string())
print('\nDiebold-Mariano:', json.dumps(reg['dm_tests'], indent=2))
print('\nGo/No-Go:', json.dumps(reg['go_no_go'], indent=2, ensure_ascii=False))
print('\nVEREDITO:', reg['veredito'])

In [ ]:
oos = pd.read_csv('../results/walkforward_oos.csv', parse_dates=['date']).set_index('date')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
oos[['y_true_bps','pred_ridge']].plot(ax=ax[0], marker='.', lw=.6, title='OOS: resíduo real vs previsto (bps)')
ax[0].axhline(0, color='k', lw=.5)
(oos['y_true_bps']-oos['pred_ridge']).abs().rolling(15).mean().plot(ax=ax[1], label='ridge')
(oos['y_true_bps']-oos['pred_b2']).abs().rolling(15).mean().plot(ax=ax[1], label='b2 (MM20)')
ax[1].set_title('MAE móvel 15d (bps)'); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Modelo final e coeficientes

In [ ]:
meta = json.load(open('../models/model_meta.json', encoding='utf-8'))
print(json.dumps(meta, indent=2, ensure_ascii=False))

## 5. Conclusão

**Veredito formal do walk-forward: NO-GO** nos critérios estritos do consenso
(melhora de MAE +8% < 15%; DM p=0.19; instabilidade entre metades). Entregáveis válidos:

1. **dolar_cc às 8:50 + ajuste médio (+5 bps)** — melhor estimativa pontual da abertura.
2. **Classificador de regime** (`model_regime.pkl`) — P(basis anormal), dimensiona a confiança do dia.
3. **Sinal direcional do ridge** — 76% de acerto em 29 dias acionáveis OOS; promissor mas N pequeno →
   **paper trading por 30-60 dias** via `05_predict_premarket.py` (loga tudo em `predictions_log.csv`).

**USDT/BRL**: reprovado no teste de 3 passos e na ablação (+4.1% MAE) — fora do modelo default,
conforme ressalva do usuário. **PIB real**: no dataset por data de publicação, apenas informativo +
dummy de evento (consenso). Ver `README.md` para o protocolo completo.